# g4-paper-graph-benchmark — Gemma 4 Developer Agent Paper Track Reproducibility


Self-contained reproducibility notebook. Prints all paper tables (Tables 1, 2A, 2B, 2C, 2D, 3) and the statistical noise model in a single ~3-minute CPU run. **Zero external dependencies** — standard library `print` only.


In [ ]:
print("=" * 72)
print("TABLE 1 — Live Leaderboard Ablations (n = 58 tasks per run)")
print("=" * 72)
print(f"{'Ref':<10} {'Exp':<22} {'Model':<5} {'Score':>6} {'Tasks':>6} {'SE':>6}  Config delta vs baseline")
print("-" * 72)
rows = [
    ("56793672","exp01","31B",0.05,2.90,0.029,"Baseline v1: thinking on, 6 min / 60 calls"),
    ("57037122","exp03","31B",0.17,9.86,0.049,"Public 0.17 replica; 6 tools (+3 graph)"),
    ("57553805","exp05","31B",0.20,11.60,0.053,"Pre-submit patch integrity check (git add -N -f)"),
    ("57654236","exp10","31B",0.17,9.86,0.049,"STATE header, 6 tools, max_output 6144"),
    ("57736284","exp12","31B",0.17,9.86,0.049,"Near-exact exp05 with 9-min timing buffer"),
    ("57836949","E0","31B",0.17,9.86,0.049,"Control: byte-identical exp05 rebuild (0.20 -> 0.17)"),
    ("57806053","e4b-v2","E4B",0.05,2.90,0.029,"Temp 0.1, imperative prompt <=2500 chars"),
    ("57889741","e4b-v4","E4B",0.05,2.90,0.029,"E4B + swe-toolkit v4; offload hypothesis falsified"),
]
for ref, exp, model, score, tasks, se, delta in rows:
    print(f"{ref:<10} {exp:<22} {model:<5} {score:>6.2f} {tasks:>6.2f} {se:>6.3f}  {delta}")
print()
print("Pooled 31B rate: 79 / 464 = 0.1702 (Pooled SE = 0.0174, n = 8 runs)")
print("E0 byte-identical control scored 0.17 -> exp05's 0.20 was an upper-tail draw (+1.6 SD), not a capability gain.")
print()


## Table 2A & 2B — Regime A: Static Graph Corpus (125/129 tasks)
Retrieval over competition AST graphs (`graphs/*.json`) and 256-dim embeddings (`embeddings/*.npz`).


In [ ]:
print("=" * 72)
print("TABLE 2A — Regime A Static Graph Corpus: File Recall Fraction (125/129 tasks)")
print("=" * 72)
print(f"{'Method':<24} {'R@1':>7} {'R@5':>7} {'R@10':>7} {'MRR':>7}")
print("-" * 72)
regime_a = [
    ("M1: Lexical Grep", 0.0504, 0.2284, 0.3036, 0.2212),
    ("M2: Dense Embedding", 0.0583, 0.2163, 0.3810, 0.2059),
    ("M3: 1-Hop AST Graph", 0.0504, 0.2265, 0.2846, 0.2103),
    ("M4: swe_toolkit (BM25)", 0.2661, 0.4350, 0.4587, 0.4354),
    ("M5: Hybrid RRF", 0.2466, 0.4745, 0.5701, 0.4332),
]
for m, r1, r5, r10, mrr in regime_a:
    print(f"{m:<24} {r1:>7.4f} {r5:>7.4f} {r10:>7.4f} {mrr:>7.4f}")
print()
print("Per-repo Regime A File R@5: M4 FastAPI 0.4791 / Rich 0.2801; M5 requests 0.8846;")
print("traceback M4/M5 0.8162/0.8547 vs dense 0.3632.")
print()
print("Regime A notes: M3 raw graph expansion degrades vs lexical grep (R@10 0.3036 -> 0.2846);")
print("80.9% of 1-hop neighborhoods exceed the 5,000-char tool cap; LLMs emit bare symbols")
print("not qualified dot-paths (0/31 gold hits, Discussion #746250).")
print()


## Table 2C & 2D — Regime B: Live Workspace Indexer (129/129 snapshots, 60,483 files)
Live repository AST indexing with `/tests/` path demotion and deterministic tie-breaking.


In [ ]:
print("=" * 72)
print("TABLE 2C — Regime B Live Workspace File Recall Ablation (129/129 snapshots)")
print("=" * 72)
print(f"{'Engine Variant':<34} {'R@1':>7} {'R@3':>7} {'R@5':>7} {'R@10':>7} {'Zero-Ret':>9}  Mechanism")
print("-" * 72)
regime_b = [
    ("base (BM25 + AST)", 0.419, 0.628, 0.713, 0.829, 0.171, "Base AST + lexical scoring"),
    ("v3 (identifier boost)", 0.419, 0.628, 0.729, 0.822, 0.178, "Path boost (hurts fastapi -0.059)"),
    ("v4 (test demotion + tie-break)", 0.419, 0.651, 0.744, 0.829, 0.171, "Reverted boost, test demotion, qualname tie-break"),
]
for m, r1, r3, r5, r10, zr, mech in regime_b:
    print(f"{m:<34} {r1:>7.3f} {r3:>7.3f} {r5:>7.3f} {r10:>7.3f} {zr:>9.3f}  {mech}")
print()
print("HEADLINE: live git-tracked source indexing + test-path demotion beats static graphs:")
print("  File R@5: 0.4350 (Regime A M4) -> 0.7440 (Regime B v4)  [+30.9 pp, +71.0% relative]")
print("  File R@10: 0.4587 -> 0.8290  [+37.0 pp]")
print()
print("Symbol re-ranking (121 gold-symbol tasks), v4 -> v6:")
print(f"{'Variant':<30} {'@1':>6} {'@3':>6} {'@5':>6} {'@10':>6}  Change")
print("-" * 72)
syms = [
    ("v4 shipped baseline", 0.157, 0.231, 0.306, 0.364, "Shipped _index.py ranking"),
    ("v5 top-8 span scoring", 0.190, 0.322, 0.388, 0.488, "+0.235s"),
    ("v5AB member-over-container", 0.215, 0.322, 0.388, 0.455, "Boost methods over classes"),
    ("v6 call-site scoring", 0.215, 0.331, 0.405, 0.463, "+0.0990 @5 net, +1.31s"),
]
for m, a1, a3, a5, a10, ch in syms:
    print(f"{m:<30} {a1:>6.3f} {a3:>6.3f} {a5:>6.3f} {a10:>6.3f}  {ch}")
print()
print("Any-Gold multi-file recall (38 tasks): Top-3 0.763, Top-5 0.816, Top-10 0.895.")
print("Strict All-Gold: Top-3 0.079, Top-5 0.079, Top-10 0.158.")
print()


## Table 3 — Gold Patch Anatomy & Playbook Precision (129 tasks)


In [ ]:
print("=" * 72)
print("TABLE 3 — Gold Patch Anatomy (129 tasks)")
print("=" * 72)
print("Median 1 file (mean 2.16) | Median 8 added lines (mean 138.6)")
print("70.5% single-file | 29.5% multi-file | 8.5% add new files (FastAPI 14.9%)")
print()
print(f"{'Playbook':<22} {'Triggers':>9} {'Gold':>6} {'Precision':>10}")
print("-" * 72)
playbooks = [
    ("RICH_RENDERING", 34, 27, "79.4%"),
    ("FASTAPI_PYDANTIC", 40, 20, "50.0%"),
    ("FALSINESS_TRAP", 21, 6, "28.6%"),
    ("HTTP_URL_COOKIES", 37, 9, "24.3%"),
    ("TYPE_SUBCLASS", 68, 16, "23.5%"),
    ("MUTABLE_STATE", 38, 5, "13.2%"),
    ("EXCEPTION_CHAIN", 15, 1, "6.7%"),
    ("OTHER_RULES", 5, 0, "0.0%"),
]
for pb, trig, gold, prec in playbooks:
    print(f"{pb:<22} {trig:>9} {gold:>6} {prec:>10}")
print(f"{'Total':<22} {258:>9} {84:>6} {'32.6%':>10}")
print()
print("swe-toolkit verify gate: treats ModuleNotFoundError for inline_snapshot / dirty_equals")
print("as a pre-existing sandbox gap, never a false-negative patch revert.")
print()


## Section 5 — Cross-Scale Statistical Noise Model


In [ ]:
print("=" * 72)
print("SECTION 5 — Within-Split SD vs Cross-Split SE (58-task public split)")
print("=" * 72)
print()
print("Within-Split Stochasticity (fixed 58 tasks):")
print("  sigma^2 = sum_i p_i(1-p_i),  sigma_within approx 0.89-1.5 tasks")
print("  exp05-family tasks [12,10,10,10,10] -> sample SD s = 0.894 tasks (~0.015 score)")
print()
print("Cross-Split Generalization Error (new sample of 58 hidden tasks):")
print("  SE_split = sqrt(p(1-p)/58) = sqrt(0.179*0.821/58) approx 0.0503 (~3.0 tasks)")
print()
print("E0 Byte-Identical Control:")
print("  exp05 0.20 (12 tasks) vs E0 byte-identical rebuild 0.17 (10 tasks) -> upper-tail draw, not a gain")
print("  Pooled 31B rate = 79/464 = 0.1702 (Pooled SE = 0.0174, n = 8 runs)")
print("  => Any leaderboard delta <= 2 tasks (<= 0.034) is statistically unresolvable on the public LB.")
print()
print("External Census (dariushafshar/gemma-4-agent-public-config-draws):")
print("  141 scored draws | 67 notebooks | 21 bundle hashes | max 0.22 (13 tasks) | no score >= 0.24")
print("  most-run bundle hash recurs in 10 notebooks scoring 0.13-0.18 (cluster ~0.17 = 10/58)")
print("  identical bundles score differently run to run -> run-to-run noise")
print()
print("4B Localization-Offload Hypothesis — FALSIFIED:")
print("  e4b_v4_swe_toolkit (ref 57889741) scored 0.05 (3 tasks)")
print("  exp14 (31B + swe-toolkit skill, 9 tools) scored 0.15 vs exp05 (6 tools) 0.20/0.17")
print("  SkillToolset injects 4 extra schemas per turn; burns 2-4 turns on skill invocation")
print("  => schema tax and edit fidelity dominate when localization is an external multi-schema tool")
print()
print("=" * 72)
print("ALL TABLES PRINTED. Zero external dependencies. End of notebook.")
print()
